# Dask on slurm-clusters

## dask.jobqueue

Dask provides functionality to interact with the slurm job scheduler via dask_jobqueue. In theory, this enables you to automatically submit, manage, and scale slurm jobs directly from within your python script. If you would like to follow this dark art, take a look at the official documentation https://jobqueue.dask.org/en/latest/  (and update the page!). 

This functionality has not been (successfully) tested yet by this author. 

## dask.distributed

You can, however, use dask as you would on your local computer. It is especially useful to easily parallelize tasks and leverage the abundance of CPU cores on the cluster. Here’s a short mock example, which should run on your local machine and a HPC facility. 

### Example slurm script

In [ ]:
## run-dask.sbatch
## run with `sbatch run-dask.sbatch`
#!/usr/bin/bash -l
#SBATCH -J dask-test
#SBATCH -o ./jobscripts/%A.out
#SBATCH -e ./jobscripts/%A.err 

#SBATCH --cpus-per-task 5
#SBATCH --mem 2000MB
#SBATCH --time 00:05:00


module purge
module load anaconda/3/2023.03
conda activate <my_dask_env>

# Run parallelized file operations
srun python run-dask.py --pause 10 --n 50 --base_path "./tmp"

### python script

In [ ]:

# run-dask.py

import logging
import os
import shutil
import time

import dask
from dask.distributed import Client

logger = logging.Logger(__name__)
formatter = logging.Formatter("%(asctime)s | [%(levelname)s] | %(message)s")
handler = logging.StreamHandler()
handler.setFormatter(formatter)
logger.addHandler(handler)


def argparser():
    parser = argparse.ArgumentParser()
    parser.add_argument("--pause", default=10, type=int, help="Pause")
    parser.add_argument("--n", default=50, type=int, help="Iterations")
    parser.add_argument("--base_path", default="./tmp", type=str, help="Write path")

    return parser.parse_args()


def run(idx: int, base_path: str = "./tmp", pause: int = 10) -> int:
    """Write a file to a temporary directory"""
    path = os.path.join(base_path, f"{idx}.txt")
    with open(path, "w") as f:
        f.write(str(idx))

    time.sleep(pause)


def main():
    # Setup, log arguments
    args = argparser()
    logger.info(f"Run with {dict(vars(args))}")
    
    if not os.path.exists(args.base_path):
        os.mkdir(args.base_path)

    # We set up our dask scheduler by calling the `dask.distributed.Client` functionality
    client = Client()
    logger.info("Run on slurm")

    logger.info("Run...")
    
    # Now, we submit our file operations to the scheduler via `.map`. 
    # This works similar to the python built-in `map` function
    # There are alternatives such as client.submit for more complicated logics
    # The results are not computed directly but stored as action directives ("futures")
    futures = client.map(lambda idx: run(idx=idx, pause=args.pause), range(args.n))
    
    # We have to explictly gather the results with client.gather
    client.gather(futures)
    logger.info("Done")

    client.close()
    
    # Clean up 
    logger.info("Wrote files {os.listdir(args.base_path)}")
    shutil.rmtree(args.base_path)
    
    logger.info("Finished")


if __name__ == "__main__":
    main()